# Bluff Detector: GPU run on Colab, from VS Code

Runs the pipeline for one model version on a free Colab T4 GPU and saves everything to your
private Hugging Face dataset repo, so nothing is lost when the Colab session ends.

**Connect first:** top right, **Select Kernel → Colab → New Colab Server → GPU → T4**.
Then set the two lines in the next cell and **Run All**. You'll be asked for two tokens.

Order of runs: `m0_smoke.yaml` first (~15 min), then `m0_base.yaml` (~1.5–2 h).
If the session drops, reconnect and **Run All** again: it picks up where it stopped.

In [ ]:
CONFIG = "configs/variants/m0_smoke.yaml"        # then "configs/variants/m0_base.yaml"
HF_REPO = "YOUR-HF-USERNAME/bluff-detector-runs"  # your private Hugging Face dataset repo

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Select Kernel -> Colab -> New Colab Server -> GPU -> T4"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Tokens (typed in, never saved in this notebook)

In [ ]:
from getpass import getpass
GH_TOKEN = getpass("GitHub token (read-only, this repo): ")
HF_TOKEN = getpass("Hugging Face token (write): ")

## 3. Get the code and install it

In [ ]:
import os, subprocess
REPO_DIR = "/content/bluff-detector"
url = f"https://{GH_TOKEN}@github.com/trevorblue/bluff-detector.git"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", url, "main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run("pip install -q -e '.[llm,data]' huggingface_hub", shell=True, check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 4. Bring back saved progress (if this version ran before)

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from bluff.config import load_config
VARIANT = load_config(CONFIG)["variant"]["id"]
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
snapshot_download(HF_REPO, repo_type="dataset", token=HF_TOKEN, local_dir=".",
                  allow_patterns=[f"data/interim/{VARIANT}/*"])
done = os.path.exists(f"data/interim/{VARIANT}/generations.jsonl")
print(VARIANT, "- resuming saved progress" if done else "- starting fresh")

def save(folders):
    for folder in folders:
        if os.path.isdir(folder):
            api.upload_folder(repo_id=HF_REPO, repo_type="dataset", folder_path=folder,
                              path_in_repo=folder, commit_message=f"{VARIANT}: {folder}")

## 5. Ask the questions (the GPU part), saving every 10 minutes

In [ ]:
import time
for stage in ("validate", "ingest", "audit"):
    subprocess.run(["bluff", stage, "--config", CONFIG], check=True)
log = open("generate.log", "w")
proc = subprocess.Popen(["bluff", "generate", "--config", CONFIG], stdout=log, stderr=log)
while proc.poll() is None:
    time.sleep(600)
    save([f"data/interim/{VARIANT}"])
    print(time.strftime("%H:%M"), "saved |", open("generate.log").read().splitlines()[-1:])
log.close()
print(open("generate.log").read()[-2000:])
assert proc.returncode == 0, "generate failed: see the log above"
save([f"data/interim/{VARIANT}"])

## 6. Label, train, evaluate, register (CPU, a few minutes)

In [ ]:
for stage in ("label", "featurize", "train", "evaluate", "register"):
    subprocess.run(["bluff", stage, "--config", CONFIG], check=True)
save([f"data/interim/{VARIANT}", "data/processed", f"artefacts/{VARIANT}",
      "artefacts/registry", f"reports/{VARIANT}"])
print(open(f"reports/{VARIANT}/summary.md").read())

## 7. On your laptop

Everything is now in your Hugging Face repo. Download it into the project folder with:

```
hf download YOUR-HF-USERNAME/bluff-detector-runs --repo-type dataset --local-dir .
```